## Query Generation

### Get random sample of urls

In [ ]:
import os
import json
import random
import pandas as pd
import re
from src.data_processing.cards import CardType, CardInfo
from src.data_processing.generate_schemas import generate_card_type_html_schema

# Define file path
simple_query_filepath = 'queries/simple_reference_based_queries.csv'

# Check if file already exists
if os.path.exists(simple_query_filepath):
    print(f"File already exists at {simple_query_filepath}")
    print("Skipping script execution to avoid overwriting existing data.")
else:
    # Generate card type schema
    card_type_schema = generate_card_type_html_schema()

    # Save schema to file
    filename = '../data/schemas/'
    schema_filename = os.path.join(filename, 'card_type_schema.json')
    os.makedirs(filename, exist_ok=True)

    with open(schema_filename, 'w', encoding='utf-8') as f:
        json.dump(card_type_schema, f, indent=4)

    # Function to clean card names for URLs
    def clean_name_for_url(name: str) -> str:
        """Clean card name for use in URLs by replacing spaces with underscores"""
        return re.sub(r'\s+', '_', name)

    # Set base URL
    base_url = "https://wildfrostwiki.com"

    # Create card infos
    card_infos = []
    for card_type, cards in card_type_schema.items():
        if card_type == 'leaders':
            continue

        for card_name in cards:
            cleaned_name = clean_name_for_url(card_name)
            card_info = CardInfo(
                card_name=card_name,
                card_type=CardType(card_type),
                card_url=f'{base_url}/{cleaned_name}'
            )
            card_infos.append(card_info)

    # Extract URLs
    urls = [card.card_url for card in card_infos]

    # Get 100 random URLs (without replacement)
    random_urls = random.sample(urls, k=min(100, len(urls)))

    # Create new DataFrame
    df = pd.DataFrame({
        'query_id': range(1, len(random_urls) + 1),
        'query': [''] * len(random_urls),  # Empty strings for now
        'ground_truth': [''] * len(random_urls),  # Empty strings for now
        'doc_reference': random_urls,  # Raw URLs
        'openAI_zero_shot': [''] * len(random_urls),  # Empty strings for now
        'openAI_RAG_response': [''] * len(random_urls)  # Empty strings for now
    })

    # Save to CSV
    os.makedirs(os.path.dirname(simple_query_filepath), exist_ok=True)
    df.to_csv(simple_query_filepath, index=False)

    print(f"Total URLs available: {len(urls)}")
    print(f"Sampled: {len(random_urls)}")
    print(f"Created DataFrame with {len(df)} rows")
    print(f"\nFirst few rows:")
    print(df.head(10))
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFile saved to: {simple_query_filepath}")

### Send to OpenAI

In [ ]:
import asyncio
import pandas as pd
from openai import AsyncOpenAI
from typing import List
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='configs/.env')

from prompts.system_prompt import SYSTEM_PROMPT

In [ ]:
# Initialize the async OpenAI client
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
async def get_zero_shot_response(query: str, model: str = "gpt-4.1-nano", seed: int = 42) -> str:
    """
    Get a zero-shot response from OpenAI for a single query.
    
    Args:
        query: The question to ask
        model: The OpenAI model to use (default: gpt-4.1-nano)
        seed: Random seed for deterministic responses (default: 42)
    
    Returns:
        The model's response as a string
    """
    try:
        response = await client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query}
            ],
            temperature=0.0,  # Use 0 for deterministic responses
            seed=seed  # Add seed for reproducibility
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error processing query: {query[:50]}... - {str(e)}")
        return f"ERROR: {str(e)}"

async def process_queries_batch(df: pd.DataFrame, batch_size: int = 25, seed: int = 42) -> pd.DataFrame:
    """
    Process all non-NA queries in the dataframe asynchronously in batches.
    Only processes rows where openAI_zero_shot is empty or NA.
    
    Args:
        df: DataFrame with 'query' column
        batch_size: Number of concurrent requests (default: 25)
        seed: Random seed for deterministic responses (default: 42)
    
    Returns:
        DataFrame with 'openAI_zero_shot' column filled
    """
    # Filter to only rows with non-empty queries AND empty/NA zero_shot responses
    valid_mask = (
        df['query'].notna() & 
        (df['query'] != '') & 
        (df['openAI_zero_shot'].isna() | (df['openAI_zero_shot'] == ''))
    )
    valid_indices = df[valid_mask].index.tolist()
    queries = df.loc[valid_mask, 'query'].tolist()
    
    print(f"Found {len(queries)} queries to process (skipping already processed rows)")
    
    if len(queries) == 0:
        print("No queries to process! All rows either have no query or already have responses.")
        return df
    
    responses = []
    
    # Process in batches to avoid rate limits
    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        print(f"Processing batch {i//batch_size + 1}/{(len(queries)-1)//batch_size + 1}")
        
        # Create tasks for this batch - using gpt-4o-mini
        tasks = [get_zero_shot_response(query, model="gpt-4o-mini", seed=seed) for query in batch]
        
        # Wait for all tasks in this batch to complete
        batch_responses = await asyncio.gather(*tasks)
        responses.extend(batch_responses)
        
        # Optional: Add a small delay between batches to be nice to the API
        if i + batch_size < len(queries):
            await asyncio.sleep(1)
    
    # Update only the rows with valid queries
    df.loc[valid_indices, 'openAI_zero_shot'] = responses
    return df

async def main():
    """Main function to load data, process queries, and save results."""
    # Load the CSV
    filepath = 'queries/simple_reference_based_queries.csv'
    df = pd.read_csv(filepath)
    
    print(f"Loaded {len(df)} total rows")
    
    # Count existing responses
    existing_responses = df['openAI_zero_shot'].notna() & (df['openAI_zero_shot'] != '')
    print(f"Already have {existing_responses.sum()} responses")
    
    # Process all queries with seed for reproducibility using gpt-4o-mini
    df = await process_queries_batch(df, batch_size=25, seed=42)
    
    # Save the updated dataframe
    df.to_csv(filepath, index=False)
    print(f"\nCompleted! Results saved to {filepath}")
    print(f"\nSample results:")
    print(df[['query', 'openAI_zero_shot']].head(10))

In [ ]:
await main()

### RAG Setup

In [ ]:
from prompts.system_prompt import RAG_PROMPT

In [ ]:
# Add after your existing imports and client setup
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

# Neo4j connection settings
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = os.getenv('TEST_EMBEDDING_NEO4J_USERNAME') or os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('TEST_EMBEDDING_NEO4J_PASSWORD') or os.getenv('NEO4J_PASSWORD')

# Embedding model settings
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
VECTOR_INDEX_NAME = 'document-embeddings'

# Initialize embedding model (do this once)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

In [ ]:
def get_retrieved_chunks(
    query: str,
    k: int = 5
) -> List[Dict[str, Any]]:
    """
    Retrieves the top-k most relevant document chunks from Neo4j based on a user query.
    Returns all available properties from the node.
    """
    # Step 1: Embed the user's query
    query_embedding = embedding_model.encode(query).tolist()
    
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    try:
        with driver.session() as session:
            # Step 2: Perform a vector similarity search in Neo4j using the vector index.
            # Return all properties from the node
            search_query = f"""
            CALL db.index.vector.queryNodes('{VECTOR_INDEX_NAME}', $k, $query_embedding)
            YIELD node, score
            RETURN node, score
            ORDER BY score DESC
            """
            
            results = session.run(
                search_query, 
                query_embedding=query_embedding, 
                k=k
            )
            
            # Step 3: Extract all properties from the node
            retrieved_chunks = []
            for record in results:
                node = record["node"]
                chunk_dict = {
                    "score": record["score"],
                    "text": node.get("text", ""),
                }
                # Add all other properties from the node
                for key, value in node.items():
                    if key not in ["text", "embedding"]:  # Skip embedding (not needed)
                        chunk_dict[key] = value if value is not None else ""
                retrieved_chunks.append(chunk_dict)
        
        return retrieved_chunks
    finally:
        driver.close()

In [ ]:
async def get_rag_response(query: str, model: str = "gpt-4.1-nano", seed: int = 42, k: int = 5) -> tuple[str, List[Dict[str, Any]]]:
    """
    Get a RAG response from OpenAI using retrieved chunks from Neo4j.
    
    Args:
        query: The question to ask
        model: The OpenAI model to use (default: gpt-4.1-nano)
        seed: Random seed for deterministic responses (default: 42)
        k: Number of chunks to retrieve (default: 5)
    
    Returns:
        Tuple of (response_string, retrieved_chunks_list) for metrics
    """
    try:
        # Retrieve relevant chunks from Neo4j
        retrieved_chunks = get_retrieved_chunks(query, k=k)
        
        if not retrieved_chunks:
            return ("ERROR: No relevant documents found in the database.", [])
        
        # Combine chunks into context
        context = "\n\n".join([chunk['text'] for chunk in retrieved_chunks])
        
        # Create the prompt with context using RAG_PROMPT template
        rag_prompt = RAG_PROMPT[0].format(query=query, context=context)
        
        response = await client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": rag_prompt}
            ],
            temperature=0.0,  # Use 0 for deterministic responses
            seed=seed  # Add seed for reproducibility
        )
        return (response.choices[0].message.content, retrieved_chunks)
    except Exception as e:
        print(f"Error processing RAG query: {query[:50]}... - {str(e)}")
        return (f"ERROR: {str(e)}", [])

In [ ]:
async def process_rag_queries_batch(df: pd.DataFrame, batch_size: int = 25, seed: int = 42) -> pd.DataFrame:
    """
    Process all queries in the dataframe with RAG, asynchronously in batches.
    Only processes rows where openAI_RAG_response is empty or NA.
    Stores both responses and retrieved chunks for retrieval metrics.
    
    Args:
        df: DataFrame with 'query' column
        batch_size: Number of concurrent requests (default: 25)
        seed: Random seed for deterministic responses (default: 42)
    
    Returns:
        DataFrame with 'openAI_RAG_response' and 'retrieved_chunks' columns filled
    """
    # Add columns if they don't exist
    if 'retrieved_chunks' not in df.columns:
        df['retrieved_chunks'] = ''
    
    # Filter to only rows with non-empty queries AND empty/NA RAG responses
    valid_mask = (
        df['query'].notna() & 
        (df['query'] != '') & 
        (df['openAI_RAG_response'].isna() | (df['openAI_RAG_response'] == ''))
    )
    valid_indices = df[valid_mask].index.tolist()
    queries = df.loc[valid_mask, 'query'].tolist()
    
    print(f"Found {len(queries)} queries to process with RAG (skipping already processed rows)")
    
    if len(queries) == 0:
        print("No queries to process! All rows either have no query or already have RAG responses.")
        return df
    
    responses = []
    all_retrieved_chunks = []
    
    # Process in batches to avoid rate limits
    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        print(f"Processing RAG batch {i//batch_size + 1}/{(len(queries)-1)//batch_size + 1}")
        
        # Create tasks for this batch
        tasks = [get_rag_response(query, seed=seed) for query in batch]
        
        # Wait for all tasks in this batch to complete
        batch_results = await asyncio.gather(*tasks)
        
        # Separate responses and chunks
        batch_responses = [result[0] for result in batch_results]
        batch_chunks = [result[1] for result in batch_results]
        
        responses.extend(batch_responses)
        all_retrieved_chunks.extend(batch_chunks)
        
        # Optional: Add a small delay between batches
        if i + batch_size < len(queries):
            await asyncio.sleep(1)
    
    # Update only the rows with valid queries
    df.loc[valid_indices, 'openAI_RAG_response'] = responses
    
    # Store retrieved chunks as JSON strings for easy parsing later
    import json
    df.loc[valid_indices, 'retrieved_chunks'] = [
        json.dumps(chunks) if chunks else '' 
        for chunks in all_retrieved_chunks
    ]
    
    return df

In [ ]:
async def main_rag():
    """Main function to load data, process queries with RAG, and save results."""
    # Load the CSV
    filepath = 'queries/simple_reference_based_queries.csv'
    df = pd.read_csv(filepath)
    
    print(f"Loaded {len(df)} total rows")
    print(f"Columns: {df.columns.tolist()}")
    
    # Initialize the RAG response column if it doesn't exist
    if 'openAI_RAG_response' not in df.columns:
        df['openAI_RAG_response'] = ''
        print("Created 'openAI_RAG_response' column")
    
    # Count existing responses
    existing_responses = df['openAI_RAG_response'].notna() & (df['openAI_RAG_response'] != '')
    print(f"Already have {existing_responses.sum()} RAG responses")
    
    # Check how many valid queries we have
    valid_queries = df['query'].notna() & (df['query'] != '')
    print(f"Found {valid_queries.sum()} valid queries")
    
    # Check how many need processing
    needs_processing = valid_queries & (df['openAI_RAG_response'].isna() | (df['openAI_RAG_response'] == ''))
    print(f"Need to process {needs_processing.sum()} queries")
    
    # Process all queries with RAG using seed for reproducibility
    df = await process_rag_queries_batch(df, batch_size=25, seed=42)
    
    # Save the updated dataframe
    df.to_csv(filepath, index=False)
    print(f"\nCompleted! Results saved to {filepath}")
    print(f"\nSample results:")
    print(df[['query', 'openAI_RAG_response']].head(10))
    
    # Show sample retrieved chunks info
    if 'retrieved_chunks' in df.columns:
        chunks_with_data = df['retrieved_chunks'].notna() & (df['retrieved_chunks'] != '')
        print(f"\nRetrieved chunks stored for {chunks_with_data.sum()} queries")
        
        # Only show example if we have data
        if chunks_with_data.any():
            import json
            sample_chunks = df[chunks_with_data]['retrieved_chunks'].iloc[0]
            parsed = json.loads(sample_chunks)
            print(f"Example: {len(parsed)} chunks retrieved for first query")
            if parsed:
                print(f"  First chunk score: {parsed[0]['score']:.4f}")
        else:
            print("No retrieved chunks data available yet")

In [ ]:
import json

def parse_retrieved_chunks(chunks_json: str) -> List[Dict[str, Any]]:
    """
    Parse retrieved chunks from JSON string stored in DataFrame.
    
    Args:
        chunks_json: JSON string of retrieved chunks
    
    Returns:
        List of chunk dictionaries with 'text' and 'score' keys
    """
    if not chunks_json or chunks_json == '':
        return []
    try:
        return json.loads(chunks_json)
    except:
        return []

In [ ]:
await main_rag()

### GUI

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from dataclasses import dataclass
import time

@dataclass
class DisplayConfig:
    """Configuration for display font sizes"""
    query_label_size: int = 16
    query_text_size: int = 14
    ground_truth_label_size: int = 16
    ground_truth_text_size: int = 14
    doc_reference_label_size: int = 16
    doc_reference_text_size: int = 14
    openai_response_label_size: int = 16
    openai_response_text_size: int = 14
    chunk_text_size: int = 14

class QueryAnnotationGUI:
    # Define available versions and their column mappings
    VERSION_CONFIGS = {
        'zero_shot': {
            'response_column': 'openAI_zero_shot',
            'validation_column': 'openAI_zero_shot_validation',
            'open_coding_column': 'openAI_zero_shot Open Coding',
            'axial_coding_column': 'openAI_zero_shot Axial Coding',
            'display_name': 'Zero-Shot',
            'chunks_column': None  # No chunks for zero-shot
        },
        'rag': {
            'response_column': 'openAI_RAG_response',
            'validation_column': 'openAI_RAG_validation',
            'open_coding_column': 'openAI_RAG Open Coding',
            'axial_coding_column': 'openAI_RAG Axial Coding',
            'display_name': 'RAG',
            'chunks_column': 'retrieved_chunks'
        }
    }
    
    def __init__(self, filepath='queries/simple_reference_based_queries.csv', config: DisplayConfig = None):
        self.filepath = filepath
        self.df = pd.read_csv(filepath)
        self.config = config if config else DisplayConfig()
        
        # Initialize columns for all versions
        for version_config in self.VERSION_CONFIGS.values():
            if version_config['validation_column'] not in self.df.columns:
                self.df[version_config['validation_column']] = ''
            if version_config['open_coding_column'] not in self.df.columns:
                self.df[version_config['open_coding_column']] = ''
            if version_config['axial_coding_column'] not in self.df.columns:
                self.df[version_config['axial_coding_column']] = ''
        
        # Get list of query IDs that have non-empty queries
        self.valid_query_ids = self.df[
            self.df['query'].notna() & (self.df['query'] != '')
        ]['query_id'].tolist()
        
        self.current_index = 0
        self.current_version = 'zero_shot'  # Default version
        
        # Create widgets
        self.create_widgets()
        self.setup_callbacks()
        self.update_display()
    
    def get_current_version_config(self):
        """Get the configuration for the currently selected version"""
        return self.VERSION_CONFIGS[self.current_version]
    
    def create_widgets(self):
        """Create all GUI widgets"""
        
        # Version selector dropdown
        version_options = {
            config['display_name']: key 
            for key, config in self.VERSION_CONFIGS.items()
        }
        self.version_dropdown = widgets.Dropdown(
            options=list(version_options.keys()),
            value=list(version_options.keys())[0],
            description='Version:',
            layout=widgets.Layout(width='200px')
        )
        
        # Query ID dropdown
        self.dropdown = widgets.Dropdown(
            options=self.valid_query_ids,
            value=self.valid_query_ids[0] if self.valid_query_ids else None,
            description='Query ID:',
            layout=widgets.Layout(width='300px')
        )
        
        # Jump to next unvalidated button
        self.jump_unvalidated_button = widgets.Button(
            description='⇨ Next Unvalidated',
            button_style='primary',
            layout=widgets.Layout(width='180px')
        )
        
        # Navigation buttons
        self.prev_button = widgets.Button(
            description='◀ Previous',
            button_style='info',
            layout=widgets.Layout(width='150px')
        )
        
        self.next_button = widgets.Button(
            description='Next ▶',
            button_style='info',
            layout=widgets.Layout(width='150px')
        )
        
        # Progress label between buttons
        self.progress_label = widgets.HTML(
            value=f"<div style='text-align: center;'><b>Query 1 of {len(self.valid_query_ids)}</b></div>",
            layout=widgets.Layout(width='200px')
        )
        
        # Display widgets
        self.query_display = widgets.HTML(value='')
        self.ground_truth_display = widgets.HTML(value='')
        self.doc_reference_display = widgets.HTML(value='')
        self.openai_response_display = widgets.HTML(value='')
        
        # Metrics display (for retrieval metrics) - Changed to Output widget
        self.metrics_display = widgets.Output()
        
        # Pass/Fail/Clear buttons
        self.pass_button = widgets.Button(
            description='✓ Pass',
            button_style='success',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        self.fail_button = widgets.Button(
            description='✗ Fail',
            button_style='danger',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        self.clear_button = widgets.Button(
            description='Clear',
            button_style='warning',
            layout=widgets.Layout(width='150px', height='40px')
        )
        
        self.validation_status = widgets.HTML(value='')
        
        # Text inputs
        self.open_coding_input = widgets.Textarea(
            placeholder='Enter open coding notes...',
            description='Open Coding:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': '120px'}
        )
        
        self.axial_coding_input = widgets.Textarea(
            placeholder='Enter axial coding notes...',
            description='Axial Coding:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': '120px'}
        )
        
        # Save status indicator
        self.save_status = widgets.HTML(value='')
        
    def setup_callbacks(self):
        """Setup all event handlers"""
        self.version_dropdown.observe(self.on_version_change, names='value')
        self.dropdown.observe(self.on_dropdown_change, names='value')
        self.jump_unvalidated_button.on_click(self.on_jump_unvalidated_click)
        self.prev_button.on_click(self.on_prev_click)
        self.next_button.on_click(self.on_next_click)
        self.pass_button.on_click(self.on_pass_click)
        self.fail_button.on_click(self.on_fail_click)
        self.clear_button.on_click(self.on_clear_click)
        self.open_coding_input.observe(self.on_open_coding_change, names='value')
        self.axial_coding_input.observe(self.on_axial_coding_change, names='value')
    
    def on_version_change(self, change):
        """Handle version dropdown change"""
        # Map display name back to version key
        version_options = {
            config['display_name']: key 
            for key, config in self.VERSION_CONFIGS.items()
        }
        self.current_version = version_options[change['new']]
        self.update_display()
    
    def on_chunk_relevance_change(self, chunk_idx, is_relevant):
        """Handle chunk relevance checkbox change"""
        row_idx = self.get_current_row_index()
        version_config = self.get_current_version_config()
        relevance_col = f"{version_config['response_column']}_chunk_relevance"
        
        # Load existing relevance dict
        existing_relevance = self.df.loc[row_idx, relevance_col] if pd.notna(self.df.loc[row_idx, relevance_col]) else '{}'
        try:
            import json
            relevance_dict = json.loads(existing_relevance) if existing_relevance else {}
        except:
            relevance_dict = {}
        
        # Update relevance for this chunk
        chunk_id = f"chunk_{chunk_idx}"
        relevance_dict[chunk_id] = is_relevant
        
        # Save back to dataframe
        self.df.at[row_idx, relevance_col] = json.dumps(relevance_dict)
        self.save_to_csv()
    
    def get_current_row(self):
        """Get the current row based on selected query_id"""
        query_id = self.dropdown.value
        return self.df[self.df['query_id'] == query_id].iloc[0]
    
    def get_current_row_index(self):
        """Get the DataFrame index for current query_id"""
        query_id = self.dropdown.value
        return self.df[self.df['query_id'] == query_id].index[0]
    
    def save_to_csv(self):
        """Save DataFrame to CSV"""
        self.df.to_csv(self.filepath, index=False)
        self.show_save_status()
    
    def show_save_status(self):
        """Show a temporary save status message"""
        self.save_status.value = '<span style="color: green;">✓ Saved</span>'
    
    def find_next_unvalidated(self):
        """Find the next unvalidated query starting from current position"""
        version_config = self.get_current_version_config()
        validation_col = version_config['validation_column']
        
        # Get queries that are unvalidated (empty or NA)
        unvalidated_mask = (
            self.df['query'].notna() & 
            (self.df['query'] != '') & 
            (self.df[validation_col].isna() | (self.df[validation_col] == ''))
        )
        unvalidated_query_ids = self.df[unvalidated_mask]['query_id'].tolist()
        
        if not unvalidated_query_ids:
            return None
        
        # Find next unvalidated after current position
        current_query_id = self.dropdown.value
        for query_id in unvalidated_query_ids:
            if query_id > current_query_id:
                return query_id
        
        # If none found after current, return first unvalidated (wrap around)
        return unvalidated_query_ids[0]
    
    def format_metrics_display(self, row):
        """Format and display retrieval metrics if available - returns widget container"""
        version_config = self.get_current_version_config()
        chunks_col = version_config.get('chunks_column')
        
        if not chunks_col or chunks_col not in row.index:
            return widgets.HTML(value='<div style="padding: 10px 0;"><i>No metrics available for this version</i></div>')
        
        chunks_json = row[chunks_col]
        if pd.isna(chunks_json) or chunks_json == '':
            return widgets.HTML(value='<div style="padding: 10px 0;"><i>No retrieval data available</i></div>')
        
        try:
            import json
            chunks = json.loads(chunks_json)
            
            if not chunks:
                return widgets.HTML(value='<div style="padding: 10px 0;"><i>No chunks retrieved</i></div>')
            
            # Get row index for storing relevance annotations
            row_idx = row.name if hasattr(row, 'name') else self.get_current_row_index()
            relevance_col = f"{version_config['response_column']}_chunk_relevance"
            
            # Initialize relevance column if needed
            if relevance_col not in self.df.columns:
                self.df[relevance_col] = ''
            
            # Load existing relevance annotations
            existing_relevance = self.df.loc[row_idx, relevance_col] if pd.notna(self.df.loc[row_idx, relevance_col]) else '{}'
            try:
                relevance_dict = json.loads(existing_relevance) if existing_relevance else {}
            except:
                relevance_dict = {}
            
            # Create chunk display widgets
            chunk_widgets = []
            
            for i, chunk in enumerate(chunks):
                chunk_id = f"chunk_{i}"
                is_relevant = relevance_dict.get(chunk_id, False)
                
                # Create checkbox
                checkbox = widgets.Checkbox(
                    value=is_relevant,
                    description='Relevant',
                    layout=widgets.Layout(width='auto')
                )
                
                # Create callback
                def make_callback(idx):
                    def on_change(change):
                        self.on_chunk_relevance_change(idx, change['new'])
                    return on_change
                
                checkbox.observe(make_callback(i), names='value')
                
                # Build metadata display with ALL available fields
                metadata_parts = []
                
                # Add URL if available (from source_file or url field)
                url = chunk.get('url') or chunk.get('source_file') or chunk.get('source')
                if url:
                    metadata_parts.append(f'<b>URL:</b> <a href="{url}" target="_blank">{url}</a>')
                
                # Add headers
                if chunk.get('header1'):
                    metadata_parts.append(f'<b>Header 1:</b> {chunk["header1"]}')
                if chunk.get('header2'):
                    metadata_parts.append(f'<b>Header 2:</b> {chunk["header2"]}')
                if chunk.get('header3'):
                    metadata_parts.append(f'<b>Header 3:</b> {chunk["header3"]}')
                
                # Add any other metadata fields (excluding text and score)
                for key, value in chunk.items():
                    if key not in ['text', 'score', 'url', 'source_file', 'source', 'header1', 'header2', 'header3'] and value:
                        metadata_parts.append(f'<b>{key}:</b> {value}')
                
                metadata_html = '<div style="margin-bottom: 10px; padding: 10px; background-color: #f5f5f5; border-radius: 5px; font-size: 13px; line-height: 1.6;">' + '<br>'.join(metadata_parts) + '</div>' if metadata_parts else ''
                
                # Create text display (simple like query/response display)
                text_display = widgets.HTML(
                    value=f"""
                    <div style="padding: 10px 0;">
                        <b>Full Text:</b><br>
                        <span style="font-size: {self.config.chunk_text_size}px; line-height: 1.5;">{chunk.get('text', 'No text available')}</span>
                    </div>
                    """
                )
                
                # Create toggle button for expand/collapse
                toggle_button = widgets.Button(
                    description='▲ Collapse',
                    button_style='',
                    layout=widgets.Layout(width='120px', height='30px')
                )
                
                # Create content container (initially visible)
                content_container = widgets.VBox([
                    widgets.HTML(value=metadata_html) if metadata_html else widgets.HTML(value=''),
                    text_display
                ], layout=widgets.Layout(display='block'))
                
                # Create chunk header with title and checkbox
                chunk_header = widgets.HBox([
                    widgets.HTML(value=f'<b style="font-size: 14px;">Chunk {i+1}</b>'),
                    toggle_button,
                    checkbox
                ], layout=widgets.Layout(justify_content='flex-start', align_items='center', margin='0 0 5px 0'))
                
                # Score on its own line
                score_display = widgets.HTML(
                    value=f'<div style="padding: 5px 0; margin-bottom: 10px;"><span style="padding: 5px 10px; background-color: #e3f2fd; border-radius: 3px; font-weight: bold; color: #1976d2;">Score: {chunk.get("score", 0):.4f}</span></div>'
                )
                
                # Create toggle callback
                def make_toggle_callback(btn, container):
                    def on_toggle(b):
                        if container.layout.display == 'none':
                            container.layout.display = 'block'
                            btn.description = '▲ Collapse'
                        else:
                            container.layout.display = 'none'
                            btn.description = '▼ Expand'
                    return on_toggle
                
                toggle_button.on_click(make_toggle_callback(toggle_button, content_container))
                
                # Create chunk container
                chunk_container = widgets.VBox([
                    chunk_header,
                    score_display,
                    content_container
                ], layout=widgets.Layout(
                    margin='10px 0',
                    padding='15px',
                    border='2px solid #2196F3',
                    border_radius='8px',
                    background_color='#fafafa'
                ))
                
                chunk_widgets.append(chunk_container)
            
            # Create summary header
            summary_html = widgets.HTML(
                value=f"""
                <div style="padding: 15px; background-color: #e8f5e9; border-radius: 5px; margin-bottom: 20px;">
                    <h3 style="margin: 0 0 10px 0; color: #2e7d32;">Retrieval Summary</h3>
                    <p style="margin: 0; font-size: 14px;"><b>Total chunks retrieved:</b> {len(chunks)}</p>
                </div>
                """
            )
            
            # Coding notes section for metrics view
            coding_notes_html = widgets.HTML(value='<h3 style="margin: 20px 0 10px 0;">Coding Notes</h3>')
            
            # Create main metrics container
            metrics_container = widgets.VBox([
                summary_html,
                coding_notes_html,
                self.open_coding_input,
                widgets.HTML(value='<div style="height: 10px;"></div>'),
                self.axial_coding_input,
                widgets.HTML(value='<h4 style="margin: 30px 0 10px 0;">Retrieved Chunks:</h4>'),
                widgets.VBox(chunk_widgets)
            ])
            
            return metrics_container
            
        except Exception as e:
            import traceback
            return widgets.HTML(value=f'<div style="padding: 10px; color: red; background-color: #ffebee; border-radius: 5px;">Error parsing metrics: {str(e)}<br><pre style="margin-top: 10px; font-size: 11px;">{traceback.format_exc()}</pre></div>')
    
    def update_display(self):
        """Update all display widgets based on current selection"""
        row = self.get_current_row()
        query_id = self.dropdown.value
        version_config = self.get_current_version_config()
        
        # Update navigation
        self.current_index = self.valid_query_ids.index(query_id)
        self.progress_label.value = f"<div style='text-align: center;'><b>Query {self.current_index + 1} of {len(self.valid_query_ids)}</b></div>"
        
        # Update prev/next button states
        self.prev_button.disabled = (self.current_index == 0)
        self.next_button.disabled = (self.current_index == len(self.valid_query_ids) - 1)
        
        # Check if there are any unvalidated queries
        next_unvalidated = self.find_next_unvalidated()
        self.jump_unvalidated_button.disabled = (next_unvalidated is None)
        
        # Display query information
        query = row['query'] if pd.notna(row['query']) else '<i>No query</i>'
        self.query_display.value = (
            f'<div style="padding: 10px 0;">'
            f'<b style="font-size: {self.config.query_label_size}px;">Query:</b><br>'
            f'<span style="font-size: {self.config.query_text_size}px;">{query}</span>'
            f'</div>'
        )
        
        ground_truth = row['ground_truth'] if pd.notna(row['ground_truth']) else '<i>No ground truth</i>'
        self.ground_truth_display.value = (
            f'<div style="padding: 10px 0;">'
            f'<b style="font-size: {self.config.ground_truth_label_size}px;">Ground Truth:</b><br>'
            f'<span style="font-size: {self.config.ground_truth_text_size}px;">{ground_truth}</span>'
            f'</div>'
        )
        
        doc_ref = row['doc_reference'] if pd.notna(row['doc_reference']) else ''
        if doc_ref:
            self.doc_reference_display.value = (
                f'<div style="padding: 10px 0;">'
                f'<b style="font-size: {self.config.doc_reference_label_size}px;">Document Reference:</b><br>'
                f'<a href="{doc_ref}" target="_blank" style="font-size: {self.config.doc_reference_text_size}px;">{doc_ref}</a>'
                f'</div>'
            )
        else:
            self.doc_reference_display.value = (
                f'<div style="padding: 10px 0;">'
                f'<b style="font-size: {self.config.doc_reference_label_size}px;">Document Reference:</b><br>'
                f'<span style="font-size: {self.config.doc_reference_text_size}px;"><i>No reference</i></span>'
                f'</div>'
            )
        
        # Display response based on selected version
        response_col = version_config['response_column']
        response = row[response_col] if pd.notna(row[response_col]) and response_col in row.index else '<i>No response yet</i>'
        display_name = version_config['display_name']
        self.openai_response_display.value = (
            f'<div style="padding: 10px; background-color: #e8f4f8; border-radius: 5px; border-left: 4px solid #2196F3;">'
            f'<b style="font-size: {self.config.openai_response_label_size}px;">OpenAI {display_name} Response:</b><br>'
            f'<span style="font-size: {self.config.openai_response_text_size}px;">{response}</span>'
            f'</div>'
        )
        
        # Display metrics if available - Updated to use Output widget
        self.metrics_display.clear_output()
        with self.metrics_display:
            metrics_widget = self.format_metrics_display(row)
            display(metrics_widget)
        
        # Update validation status
        validation_col = version_config['validation_column']
        validation = row[validation_col] if validation_col in row.index else ''
        if pd.notna(validation) and validation != '':
            if validation.lower() == 'pass':
                self.validation_status.value = '<span style="color: green; font-size: 16px; font-weight: bold;">✓ PASS</span>'
            else:
                self.validation_status.value = '<span style="color: red; font-size: 16px; font-weight: bold;">✗ FAIL</span>'
        else:
            self.validation_status.value = '<span style="color: gray; font-style: italic;">Not validated yet</span>'
        
        # Update text inputs (without triggering save)
        self.open_coding_input.unobserve(self.on_open_coding_change, names='value')
        self.axial_coding_input.unobserve(self.on_axial_coding_change, names='value')
        
        open_coding_col = version_config['open_coding_column']
        axial_coding_col = version_config['axial_coding_column']
        self.open_coding_input.value = row[open_coding_col] if open_coding_col in row.index and pd.notna(row[open_coding_col]) else ''
        self.axial_coding_input.value = row[axial_coding_col] if axial_coding_col in row.index and pd.notna(row[axial_coding_col]) else ''
        
        self.open_coding_input.observe(self.on_open_coding_change, names='value')
        self.axial_coding_input.observe(self.on_axial_coding_change, names='value')
        
        # Clear save status when switching queries
        self.save_status.value = ''
    
    def on_dropdown_change(self, change):
        """Handle dropdown selection change"""
        self.update_display()
    
    def on_jump_unvalidated_click(self, button):
        """Handle jump to next unvalidated button click"""
        next_unvalidated = self.find_next_unvalidated()
        if next_unvalidated is not None:
            self.dropdown.value = next_unvalidated
    
    def on_prev_click(self, button):
        """Handle previous button click"""
        if self.current_index > 0:
            self.dropdown.value = self.valid_query_ids[self.current_index - 1]
    
    def on_next_click(self, button):
        """Handle next button click"""
        if self.current_index < len(self.valid_query_ids) - 1:
            self.dropdown.value = self.valid_query_ids[self.current_index + 1]
    
    def on_pass_click(self, button):
        """Handle pass button click"""
        idx = self.get_current_row_index()
        version_config = self.get_current_version_config()
        self.df.at[idx, version_config['validation_column']] = 'Pass'
        self.save_to_csv()
        self.update_display()
    
    def on_fail_click(self, button):
        """Handle fail button click"""
        idx = self.get_current_row_index()
        version_config = self.get_current_version_config()
        self.df.at[idx, version_config['validation_column']] = 'Fail'
        self.save_to_csv()
        self.update_display()
    
    def on_clear_click(self, button):
        """Handle clear button click"""
        idx = self.get_current_row_index()
        version_config = self.get_current_version_config()
        self.df.at[idx, version_config['validation_column']] = ''
        self.save_to_csv()
        self.update_display()
    
    def on_open_coding_change(self, change):
        """Handle open coding text change - auto-save"""
        idx = self.get_current_row_index()
        version_config = self.get_current_version_config()
        self.df.at[idx, version_config['open_coding_column']] = change['new']
        self.save_to_csv()
    
    def on_axial_coding_change(self, change):
        """Handle axial coding text change - auto-save"""
        idx = self.get_current_row_index()
        version_config = self.get_current_version_config()
        self.df.at[idx, version_config['axial_coding_column']] = change['new']
        self.save_to_csv()
    
    def display(self):
        """Display the complete GUI with tabs"""
        
        # Version selector and query selector row
        version_query_box = widgets.HBox([
            self.version_dropdown,
            self.dropdown,
            self.jump_unvalidated_button
        ], layout=widgets.Layout(justify_content='flex-start', margin='10px 0'))
        
        # Navigation row (centered with label in middle)
        nav_box = widgets.HBox([
            self.prev_button,
            self.progress_label,
            self.next_button
        ], layout=widgets.Layout(justify_content='center', margin='10px 0'))
        
        # Query information section (for main tab)
        info_box = widgets.VBox([
            self.query_display,
            widgets.HTML(value='<div style="height: 5px;"></div>'),
            self.ground_truth_display,
            widgets.HTML(value='<div style="height: 5px;"></div>'),
            self.doc_reference_display,
            widgets.HTML(value='<div style="height: 10px;"></div>'),
            self.openai_response_display
        ])
        
        # Validation section (centered)
        validation_buttons = widgets.HBox([
            self.pass_button,
            self.fail_button,
            self.clear_button,
            self.validation_status,
            self.save_status
        ], layout=widgets.Layout(justify_content='center', margin='20px 0'))
        
        # Coding inputs section
        coding_box = widgets.VBox([
            widgets.HTML(value='<h3>Coding Notes</h3>'),
            self.open_coding_input,
            widgets.HTML(value='<div style="height: 10px;"></div>'),
            self.axial_coding_input
        ])
        
        # Main tab content
        main_tab_content = widgets.VBox([
            info_box,
            widgets.HTML(value='<hr>'),
            validation_buttons,
            coding_box
        ])
        
        # Retrieval metrics tab content (coding notes now right after summary)
        metrics_tab_content = widgets.VBox([
            self.metrics_display
        ], layout=widgets.Layout(padding='10px'))
        
        # Create tabs
        tab = widgets.Tab()
        tab.children = [main_tab_content, metrics_tab_content]
        tab.set_title(0, 'Query & Response')
        tab.set_title(1, 'Retrieval Metrics')
        
        # Main container
        main_container = widgets.VBox([
            widgets.HTML(value='<h2>Query Annotation Tool</h2>'),
            version_query_box,
            nav_box,
            widgets.HTML(value='<hr>'),
            tab
        ], layout=widgets.Layout(padding='20px'))
        
        display(main_container)

In [ ]:
# Usage with default config:
gui = QueryAnnotationGUI('queries/simple_reference_based_queries.csv')
gui.display()

In [ ]:
stop

### Generate Taxonomy of failure modes

In [ ]:
import pandas as pd
import asyncio
from openai import AsyncOpenAI
import os

# Initialize OpenAI client
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

TAXONOMY_SYSTEM_PROMPT = """You are an expert at qualitative coding analysis, specifically creating axial codes from open codes.

You will be given a numbered list of open codes that describe various failure modes from an LLM evaluation.

Your task is to perform axial coding:
1. Analyze all the open codes
2. Identify common themes and patterns across the codes
3. Create higher-level axial codes (categories) that group related open codes
4. Each axial code should represent a broader conceptual category
5. Provide clear definitions for each axial code
6. Reference which open codes (by number) fall under each axial code

Format your response as a well-structured markdown document with:
- A title: "Failure Mode Taxonomy - Axial Codes"
- A brief introduction explaining the coding approach
- Axial codes as H2 headers (##)
- Sub-categories as H3 headers (###) if needed
- Clear descriptions of each axial code category
- List the relevant open code numbers that fall under each axial code
- A summary section with key insights

Be comprehensive but concise. Make the taxonomy useful for understanding and addressing these failure modes."""

async def generate_taxonomy(open_codes: list[str], model: str = "gpt-4o-mini") -> str:
    """
    Generate axial codes from open codes.
    
    Args:
        open_codes: List of open code strings
        model: The OpenAI model to use
    
    Returns:
        Markdown formatted taxonomy with axial codes
    """
    # Remove duplicates and filter out empty/NA values
    unique_codes = list(set([code for code in open_codes if pd.notna(code) and code.strip() != '']))
    
    # Sort for consistency
    sorted_codes = sorted(unique_codes)
    
    # Create a numbered list of codes
    codes_text = "\n".join([f"{i+1}. {code}" for i, code in enumerate(sorted_codes)])
    
    user_message = f"""Here are the open codes from the failure analysis:\n\n{codes_text}\n\n
Please create axial codes that group these open codes into higher-level categories. Reference the open codes by their numbers."""
    
    try:
        response = await client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": TAXONOMY_SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            temperature=0.3,  # Slightly higher for more creative categorization
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error generating taxonomy: {str(e)}")
        return f"ERROR: {str(e)}"

async def codes():
    """Main function to load open codes, generate taxonomy, and save results."""
    # Load the CSV
    filepath = 'queries/simple_reference_based_queries.csv'
    df = pd.read_csv(filepath)
    
    print(f"Loaded {len(df)} total rows")
    
    # Check if Open Coding column exists
    column_name = 'openAI_zero_shot Open Coding'
    if column_name not in df.columns:
        print(f"ERROR: '{column_name}' column not found in the CSV")
        print(f"Available columns: {df.columns.tolist()}")
        return
    
    # Get all open codes
    open_codes = df[column_name].tolist()
    valid_codes = [code for code in open_codes if pd.notna(code) and str(code).strip() != '']
    
    print(f"Found {len(valid_codes)} open codes")
    print(f"Unique codes: {len(set(valid_codes))}")
    
    if len(valid_codes) == 0:
        print("No open codes found to process!")
        return
    
    # Check if file already exists BEFORE making API call
    output_path = 'queries/simple_reference_based_failure_mode_taxonomy.md'
    
    if os.path.exists(output_path):
        print(f"\n⚠️  File already exists at {output_path}")
        print("Skipping taxonomy generation to avoid overwriting existing file.")
        print("Delete or rename the existing file if you want to generate a new taxonomy.")
        return
    
    print("\nGenerating axial codes from open codes...")
    taxonomy = await generate_taxonomy(open_codes)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(taxonomy)
    
    print(f"\n✓ Axial codes taxonomy saved to {output_path}")


In [ ]:
await codes()

### Basic Stats for Basic Cards

In [ ]:
import random

In [ ]:
card_info_file_path = "data/structured_outputs"

In [ ]:
stats_query = "What are the stats for {card}?"

In [ ]:
import os

files = os.listdir(card_info_file_path)

In [ ]:
files

In [ ]:
queries = []
for file in files:
    
    cards = os.listdir(f'{card_info_file_path}/{file}')
    k = random.randint(2, 3)
    
    random_sample = random.sample(cards, k=k)
    random_sample = [c.split('.html')[0] for c in random_sample]
    print(random_sample)

    for r_s in random_sample:

        queries.append(stats_query.format(card=r_s))

In [ ]:
len(queries)

### Tribes

In [ ]:
tribe_query = "What tribe does {card} belong to?"

In [ ]:
"What cards can belong to any tribe?"


In [ ]:
tribe_exclusive_query = "What cards are exclusive to the {tribe_name} tribe?"

In [ ]:
tribe_exclusive_item_query = "What items are exclusive to the {tribe_name} tribe?"

### Abilities

In [ ]:
ability_query = "What ability does {card} have?"

### Relational Questions

In [ ]:
"What cards have only health and counter stats?"
"What cards have only scrap and counter stats?"
"What cards have only scrap and attack stats?"
"What cards have only health, counter, and attack stats?"
"What cards have only health and attack stats?"

In [ ]:
"What card is both an enemy and a companion card?"

In [ ]:
"What cards are can be both a player and enemy clunker?"

In [ ]:
"What fights can Gobling spawn in?"
"What fights can Gobling NOT spawn in?"

In [ ]:
from neo4j import GraphDatabase
import os 
from dotenv import load_dotenv
load_dotenv(dotenv_path="configs/.env")

# Set up connection details
uri = "bolt://localhost:7687"  # update if using cloud or custom port
username = os.getenv('NEO4J_USERNAME')          # update with actual username
password = os.getenv('NEO4J_PASSWORD')          # update with actual password
driver = GraphDatabase.driver(uri, auth=(username, password))

query = """
MATCH (c:Card)
RETURN (c) AS card
"""

with driver.session() as session:
    result = session.run(query)
    card_dicts = [dict(record["card"]) for record in result]
        


In [ ]:
card_dicts

In [ ]:
def create_stats_queries(card_dicts, stats_query_template="In Wildfrost, what are the stats for {card}?"):
    keys_of_interest = ["health", "counter", "scrap", "attack"]
    queries = []

    for card in card_dicts:
        card_name = card.get("card_name", "Unknown Card")
        query_text = stats_query_template.format(card=card_name)
        
        # Extract stats if present
        ground_truth = {key: card[key] for key in keys_of_interest if key in card}

        if not ground_truth:
            ground_truth = "No stats for this card"
        else:
            # Add prefix text to the stats output
            stats_text = ", ".join(f"{key}: {value}" for key, value in ground_truth.items())
            ground_truth = "The stats are:\n" + stats_text

        queries.append({
            "query": query_text,
            "ground_truth": ground_truth
        })
    return queries


In [ ]:
query_list = create_stats_queries(card_dicts)
for item in query_list:
    print(item)

In [ ]:
import asyncio
import json
from ollama import AsyncClient

async def query_single(card_query, model_name="gemma3", client=None):
    query_text = card_query["query"]
    ground_truth = card_query["ground_truth"]
    system_prompt = (
        "You are a helpful assistant that only uses the provided information to answer queries.\n"
        "Answer based only on this data. Ensure you provide a full answer to the query."
    )

    ground_truth_str = json.dumps(ground_truth)

    response = await client.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query_text + "\n" + ground_truth_str},
        ],
    )
    print(f"Query: {query_text}")
    print(f"Response: {response['message']['content']}")
    print("-" * 40)

async def query_ollama_model_async(card_queries, model_name="gemma3"):
    client = AsyncClient()
    tasks = [
        query_single(card_query=item, model_name=model_name, client=client)
        for item in card_queries
    ]
    await asyncio.gather(*tasks)


In [ ]:
# Example usage:
card_queries = create_stats_queries(card_dicts)


In [ ]:
len(card_queries)

In [ ]:
await query_ollama_model_async(card_queries)